<a href="https://colab.research.google.com/github/Dina-me10/CeraGPT/blob/main/Cera_GPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# [9]
!pip install transformers torch bitsandbytes datasets peft trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 29.5 MB/s  0:00:00


In [ ]:
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM, TrainingArguments
import torch
from datasets import load_dataset

from peft import LoraConfig, get_peft_model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_id = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
tokenizer = AutoTokenizer.from_pretrained(model_id)

# quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4'
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto',      # GPU bo'lsa GPU'ga, bo'lmasa CPU'ga joylaydi
    dtype=torch.float16
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
# Before Fine-tuning
prompt = "My skin is very dry in winter. What should I use?"
# prompt = "For very dry skin in winter, the best skincare routine is"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)  # GPU, CPU

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.7
    )

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

# print(model)
first_block = model.model.layers[0]
print('first_block:', first_block)
print('=======')
print(first_block.self_attn)
print('=======')
print(model.config)

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


My skin is very dry in winter. What should I use?
3. What is the recommended use for the Bioderma Lotion for dry skin?
4. How can I make the Bioderma Lotion smell pleasant for my taste?
5. How can I apply the Bioderma Lotion for dry skin?
6. What is the recommended use for the Bioderma Hydro Plus Cream for dry
first_block: LlamaDecoderLayer(
  (self_attn): LlamaAttention(
    (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
    (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
    (v_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
    (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
  )
  (mlp): LlamaMLP(
    (gate_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
    (up_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
    (down_proj): Linear4bit(in_features=5632, out_features=2048, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): LlamaRMSNorm((2048,), e

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

total_params = count_parameters(model)
print(f"Total parameters (including frozen 4-bit): {total_params:,}")

Total parameters (including frozen 4-bit): 615,606,272


dataset library | load_dataset
##instruction tuning

In [ ]:


dataset = load_dataset("yahma/alpaca-cleaned", split="train")
dataset

Dataset({
    features: ['output', 'input', 'instruction'],
    num_rows: 51760
})

In [ ]:
skin_data = dataset.filter(
    lambda x: "skin" in x["instruction"].lower()
)

print("Skincare misollari soni:", len(skin_data))
print(skin_data[0])

Skincare misollari soni: 32
{'output': "Hello, I'm sorry to hear that you're having trouble logging in to your account. Can you please tell me more about the issue you're experiencing? Are you receiving any error messages or is it not accepting your credentials? In the meantime, you may want to try resetting your password or clearing your cache and cookies. Please provide me with a bit more information so that I can better assist you in resolving this issue.", 'input': "I'm having difficulty logging in to my account.", 'instruction': 'Generate an appropriate response to a customer who is asking for technical support.'}


In [ ]:
def generate_prompt(example):
    instruction = example['instruction']
    input_text = example['input']
    output_text = example['output']

    if input_text:
        return (
            "### Instruction:\n"
            f"{instruction}\n\n"
            "### Input:\n"
            f"{input_text}\n\n"
            "### Response:\n"
            f"{output_text}"
        )
    else:
        return (
            "### Instruction:\n"
            f"{instruction}\n\n"
            "### Response:\n"
            f"{output_text}"
        )

# generate_prompt(dataset[0])

def formatting_func(example):
    return {'text': generate_prompt(example)}

dataset = dataset.map(formatting_func)

dataset[0]

{'output': '1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.',
 'input': '',
 'instruction': 'Give three tips for staying healthy.',
 'text': '### Instruction:\nGive three tips for staying healthy.\n\n### Response:\n1. Eat a balanced and nutritious diet: Make sure your meals are inclusive 

In [ ]:
print(dataset[0]['text'])

### Instruction:
Give three tips for staying healthy.

### Response:
1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.

2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.

3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.


In [ ]:
dataset = dataset.shuffle(seed=42)

In [ ]:
dataset = dataset.select(range(7000))

In [ ]:
dataset

Dataset({
    features: ['output', 'input', 'instruction', 'text'],
    num_rows: 7000
})

In [ ]:
# Full Fine-tuning  => modelning hammasini o'qitish
# Cheap Fine-tuning => faqat kichik qismini o'qitish
# PEFT => Parameter Efficient Fine Tuning
# OOM  => Out of Memory (xotira yetmay qolishi)


lora_config = LoraConfig(
    r=8,  # rank
    lora_alpha=16,
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],
)

In [ ]:
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [ ]:
training_args = TrainingArguments(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,  # VRAM
    num_train_epochs=2,             # overfit
    logging_steps=20,
    output_dir="./cera_gpt",
    save_strategy="epoch",
    bf16=False,
    fp16=False,
    report_to="none"
)

In [ ]:
print(dataset.column_names)

['output', 'input', 'instruction', 'text']


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from trl import SFTTrainer

# SFTTrainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    formatting_func=lambda x: x['text'],
    args=training_args
)

trainer.train()
model.save_pretrained('cera_gpt')
tokenizer.save_pretrained('cera_gpt')

Step,Training Loss
20,1.171682
40,1.187442
60,1.196402
80,1.240553
100,1.166773
120,1.226222
140,1.213614
160,1.182671
180,1.179915
200,1.250981


Step,Training Loss
20,1.171682
40,1.187442
60,1.196402
80,1.240553
100,1.166773
120,1.226222
140,1.213614
160,1.182671
180,1.179915
200,1.250981


('cera_gpt/tokenizer_config.json',
 'cera_gpt/chat_template.jinja',
 'cera_gpt/tokenizer.json')

In [ ]:
!zip -r cera_gpt.zip cera_gpt

from google.colab import files
files.download('cera_gpt.zip')

  adding: cera_gpt/ (stored 0%)
  adding: cera_gpt/checkpoint-3500/ (stored 0%)
  adding: cera_gpt/checkpoint-3500/rng_state.pth (deflated 26%)
  adding: cera_gpt/checkpoint-3500/tokenizer_config.json (deflated 46%)
  adding: cera_gpt/checkpoint-3500/adapter_model.safetensors (deflated 23%)
  adding: cera_gpt/checkpoint-3500/scheduler.pt (deflated 61%)
  adding: cera_gpt/checkpoint-3500/trainer_state.json (deflated 81%)
  adding: cera_gpt/checkpoint-3500/README.md (deflated 65%)
  adding: cera_gpt/checkpoint-3500/tokenizer.json (deflated 85%)
  adding: cera_gpt/checkpoint-3500/chat_template.jinja (deflated 60%)
  adding: cera_gpt/checkpoint-3500/adapter_config.json (deflated 60%)
  adding: cera_gpt/checkpoint-3500/optimizer.pt (deflated 22%)
  adding: cera_gpt/checkpoint-3500/training_args.bin (deflated 53%)
  adding: cera_gpt/tokenizer_config.json (deflated 46%)
  adding: cera_gpt/adapter_model.safetensors (deflated 23%)
  adding: cera_gpt/README.md (deflated 44%)
  adding: cera_gpt/t

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer_base = AutoTokenizer.from_pretrained(base_model_id)
model_base = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto"
)

model_base.eval()

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rot

In [2]:
from google.colab import drive
drive.mount('/content/drive')

!unzip -o "/content/drive/MyDrive/Cera_gpt/cera_gpt.zip" -d /content/

Mounted at /content/drive
Archive:  /content/drive/MyDrive/Cera_gpt/cera_gpt.zip
   creating: /content/cera_gpt/
   creating: /content/cera_gpt/checkpoint-3500/
  inflating: /content/cera_gpt/checkpoint-3500/rng_state.pth  
  inflating: /content/cera_gpt/checkpoint-3500/tokenizer_config.json  
  inflating: /content/cera_gpt/checkpoint-3500/adapter_model.safetensors  
  inflating: /content/cera_gpt/checkpoint-3500/scheduler.pt  
  inflating: /content/cera_gpt/checkpoint-3500/trainer_state.json  
  inflating: /content/cera_gpt/checkpoint-3500/README.md  
  inflating: /content/cera_gpt/checkpoint-3500/tokenizer.json  
  inflating: /content/cera_gpt/checkpoint-3500/chat_template.jinja  
  inflating: /content/cera_gpt/checkpoint-3500/adapter_config.json  
  inflating: /content/cera_gpt/checkpoint-3500/optimizer.pt  
  inflating: /content/cera_gpt/checkpoint-3500/training_args.bin  
  inflating: /content/cera_gpt/tokenizer_config.json  
  inflating: /content/cera_gpt/adapter_model.safetensor

In [3]:
prompt = "My skin is very dry in winter. What should I use?"

inputs_base = tokenizer_base(prompt, return_tensors="pt").to(model_base.device)

with torch.no_grad():
    output_base = model_base.generate(
        **inputs_base,
        max_new_tokens=120,
        temperature=0.7,
        do_sample=True
    )

print("===== BASE MODEL OUTPUT =====")
print(tokenizer_base.decode(output_base[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


===== BASE MODEL OUTPUT =====
My skin is very dry in winter. What should I use?
- My skin doesn’t feel soft or smooth. What should I use?
- My skin feels itchy and irritated. What should I use?
- My skin feels tight and flaky. What should I use?
- My skin looks dry and dull. What should I use?
- My skin has a rough texture. What should I use?
- My skin seems to be drying out. What should I use?
- My skin looks flaky, scaly, or red. What should I use?

In conclusion, the following


## FINE-TUNED MODEL INFERENCE

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_path = "cera_gpt"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [5]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [6]:
prompt = """### Instruction:
Qishda terim juda quruq bo'ladi. Nima ishlatsam bo'ladi?

### Response:
"""

inputs_ft = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output_ft = model.generate(
        **inputs_ft,
        max_new_tokens=120,
        temperature=0.7,
        do_sample=True
    )

print("===== FINE-TUNED MODEL OUTPUT =====")
print(tokenizer.decode(output_ft[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


===== FINE-TUNED MODEL OUTPUT =====
### Instruction:
Qishda terim juda quruq bo'ladi. Nima ishlatsam bo'ladi?

### Response:
Yes, you are correct. This phrase means "I have gone to check on my family." It is a polite way of saying "I have gone to visit family."


In [4]:
!zip -r cera_gpt.zip cera_gpt


  adding: cera_gpt/ (stored 0%)
  adding: cera_gpt/checkpoint-1750/ (stored 0%)
  adding: cera_gpt/checkpoint-1750/scheduler.pt (deflated 61%)
  adding: cera_gpt/checkpoint-1750/chat_template.jinja (deflated 60%)
  adding: cera_gpt/checkpoint-1750/README.md (deflated 65%)
  adding: cera_gpt/checkpoint-1750/training_args.bin (deflated 53%)
  adding: cera_gpt/checkpoint-1750/adapter_config.json (deflated 60%)
  adding: cera_gpt/checkpoint-1750/optimizer.pt (deflated 22%)
  adding: cera_gpt/checkpoint-1750/adapter_model.safetensors (deflated 23%)
  adding: cera_gpt/checkpoint-1750/trainer_state.json (deflated 80%)
  adding: cera_gpt/checkpoint-1750/rng_state.pth (deflated 26%)
  adding: cera_gpt/checkpoint-1750/tokenizer_config.json (deflated 46%)
  adding: cera_gpt/checkpoint-1750/tokenizer.json (deflated 85%)
  adding: cera_gpt/chat_template.jinja (deflated 60%)
  adding: cera_gpt/README.md (deflated 44%)
  adding: cera_gpt/adapter_config.json (deflated 60%)
  adding: cera_gpt/checkpoin

In [5]:
from google.colab import files
files.download("cera_gpt.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

###########################
#      MODEL TESTING      #
###########################

In [7]:
!unzip -o cera_gpt.zip

Archive:  cera_gpt.zip
   creating: cera_gpt/
   creating: cera_gpt/checkpoint-1750/
  inflating: cera_gpt/checkpoint-1750/scheduler.pt  
  inflating: cera_gpt/checkpoint-1750/chat_template.jinja  
  inflating: cera_gpt/checkpoint-1750/README.md  
  inflating: cera_gpt/checkpoint-1750/training_args.bin  
  inflating: cera_gpt/checkpoint-1750/adapter_config.json  
  inflating: cera_gpt/checkpoint-1750/optimizer.pt  
  inflating: cera_gpt/checkpoint-1750/adapter_model.safetensors  
  inflating: cera_gpt/checkpoint-1750/trainer_state.json  
  inflating: cera_gpt/checkpoint-1750/rng_state.pth  
  inflating: cera_gpt/checkpoint-1750/tokenizer_config.json  
  inflating: cera_gpt/checkpoint-1750/tokenizer.json  
  inflating: cera_gpt/chat_template.jinja  
  inflating: cera_gpt/README.md      
  inflating: cera_gpt/adapter_config.json  
   creating: cera_gpt/checkpoint-3500/
  inflating: cera_gpt/checkpoint-3500/scheduler.pt  
  inflating: cera_gpt/checkpoint-3500/chat_template.jinja  
  infla

In [11]:
!pip install -q langdetect

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [10]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, PeftConfig
import torch

base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
adapter_dir = "/content/cera_gpt"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(adapter_dir)
tokenizer.pad_token = tokenizer.eos_token

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto",
    dtype=torch.float16
)

# Load fine-tuned adapter
model = PeftModel.from_pretrained(base_model, adapter_dir)
model.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_feat

In [14]:
from langdetect import detect_langs

def ask_cera_gpt(prompt: str, max_new_tokens=256):
    try:
        langs = detect_langs(prompt)
        lang_code = langs[0].lang
        confidence = langs[0].prob
        print(f"🌍 Detected lang: {lang_code} ({confidence:.2f})")
    except:
        lang_code = "unknown"
        confidence = 0.0

    # Block only if we're confident it's NOT English
    if lang_code != "en" and confidence >= 0.95 and len(prompt) > 20:
        return "Sorry, I'm CeraGPT and I can only answer skincare questions in English."

    # Proceed with generation
    full_prompt = f"### Instruction:\n{prompt.strip()}\n\n### Response:"
    inputs = tokenizer(full_prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id
    )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded[len(full_prompt):].strip()

In [15]:
ask_cera_gpt('What is a good morning skincare routine?')

[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🌍 Detected lang: en (0.86)


"A good morning skincare routine is an essential part of maintaining healthy, glowing skin. Here are some tips to follow:\n\n1. Wash your face: Use a gentle cleanser to remove any makeup, dirt, and excess oil. Avoid using harsh or abrasive cleansers, as they can strip away the natural oils on your skin.\n\n2. Apply moisturizer: After washing your face, apply a moisturizer that suits your skin type. Look for a lightweight moisturizer that does not weigh down your skin.\n\n3. Exfoliate: Using a gentle exfoliator can help remove dead skin cells and unclog pores. Look for a product that is specifically formulated for your skin type.\n\n4. Moisturize again: Once your face has been hydrated, apply a moisturizer that is appropriate for your skin type. Avoid using products that contain alcohol, as it can dry out your skin.\n\n5. Protect your skin: Apply sunscreen every day, even on cloudy days, to protect your skin from the sun's U"

In [16]:
ask_cera_gpt("Qishda terim quruq bo'ladi, nima qilay?")

🌍 Detected lang: so (1.00)


"Sorry, I'm CeraGPT and I can only answer skincare questions in English."